## Data Analysis using Google ADK
### Capstone Project Dr Sompal Singh

In [1]:
#!pip install google-adk
#!pip install pandas
#!pip install matplotlib
#!pip install --upgrade --force-reinstall "google-adk[extensions]"
#!pip uninstall scipy

In [2]:
#!pip show google-adk

In [18]:
# FOR csv_tools.py
import pandas as pd
from scipy.stats import chi2_contingency

# FOR code_tools.py
import sys
import io
import contextlib
import traceback

# FOR agent.py
import os
from google.adk.agents import Agent
from google.adk.workflow import Workflow
from google.adk.planners import PlanReActPlanner
#from csv_tools import inspect_csv

# FOR agent.py
#import os
#from google.adk.agents import Agent
#from google.adk.planners import PlanReActPlanner
#from code_tools import execute_python_code

from google.adk.tools import google_search

# FOR main.py
import asyncio
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
#from agent import data_analyst_agent

# FOR main.py
#import asyncio
#from google.genai import types
#from google.adk.sessions import InMemorySessionService
#from google.adk.runners import Runner
#from agent import graph_analyst_agent

In [19]:
import os
# Following is a sensitive Information
os.environ["GEMINI_API_KEY"] = "<your Gemini API Key>"

In [20]:
# inspect_csv.py
def inspect_csv(file_path: str) -> str:
    """
    Reads a local CSV file and returns its columns, data types, 
    and the first few rows as context for analysis.
    
    Args:
        file_path: The absolute or relative path to the CSV file.
    """
    try:
        df = pd.read_csv(file_path)
        summary = (
            f"Columns & Types:\n{df.dtypes.to_string()}\n\n"
            f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n\n"
            f"First 3 rows:\n{df.head(3).to_string()}"
        )
        return summary
    except Exception as e:
        return f"Error reading CSV file: {str(e)}"

In [21]:
# load_csv.py
def load_csv(file_path: str) -> str:
    """
    Reads a local CSV file and returns its columns, data types, 
    and the comma separated data as text for analysis.
    
    Args:
        file_path: The absolute or relative path to the CSV file.
    """
    try:
        df = pd.read_csv(file_path)
        summary = (
            f"Columns & Types:\n{df.dtypes.to_string()}\n\n"
            f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n\n"
            f"Data as text:\n{df.to_csv(index=False)}"
        )
        return summary
    except Exception as e:
        return f"Error reading CSV file: {str(e)}"

In [22]:
# do_chisquare.py
def do_chisquare(file_path: str, var1: str, var2: str) -> str:
    """
    Reads a local CSV file and returns results of 
    Chi-Square test between two variables.
    
    Args:
        file_path: The absolute or relative path to the CSV file.
        var1: Name of first categorical variable.
        var2: Name of second categorical variable.
    """
    try:
        df = pd.read_csv(file_path)
        contingency_table = pd.crosstab(df[var1], df[var2])
        chi2, p, dof, expected = chi2_contingency(contingency_table)
        summary = (
            f"ChiSquare Val:\n{chi2}\n\n"
            f"p value: {p} columns\n\n"
        )
        return summary
    except Exception as e:
        return f"Error reading CSV file: {str(e)}"

In [23]:
# csv_data_analyst_agent.py
# Load your model preference from environment variables
MODEL = os.getenv("ADK_MODEL", "gemini-3.5-flash")

# Define the data analyst agent
csv_data_analyst_agent = Agent(
    name="csv_data_analyst_agent",
    model=MODEL,
    description="An agent specialized in analyzing CSV files and generating insights.",
    instruction=(
        "You are an expert data analyst. Use the provided 'load_csv' tool "
        "to look at the metadata and content of a file. Then, use your internal "
        "reasoning capabilities to compute summaries, pinpoint patterns, and answer "
        "user business questions about the data accurately."
    ),
    tools=[load_csv],
    planner=PlanReActPlanner()  # Structurally enforces Plan -> Action -> Observation steps
)


In [24]:
# chi_square_analyst_agent.py
MODEL = os.getenv("ADK_MODEL", "gemini-3.5-flash")

chi_square_analyst_agent = Agent(
    name="chi_square_analyst_agent",
    model=MODEL,
    description="An agent that runs Python code for Chi-Square test for association bewteen two variables.",
    instruction=(
        "You are an expert data analyst. Your task is to perform Chi-Square test for association bewteen two categorical variables"
        "CRITICAL RULES:\n"
        "1. Always print p-value with the results.\n"
        "3. Use the 'do_chisquare' tool to get the results."
    ),
    tools=[do_chisquare],
    planner=PlanReActPlanner()
)

In [25]:
# root_agent.py as workflow
MODEL = os.getenv("ADK_MODEL", "gemini-3.5-flash")

#root_agent = SequentialAgent(
#    name="data_insights_and_chi_square_pipeline",
#    description="A multi-agent workflow that produce data insights and do chi-square test.",
#    sub_agents=[csv_data_analyst_agent, chi_square_analyst_agent]
#)

root_agent = Workflow(
    name="MyPipeline",
    edges = [
        ("START", csv_data_analyst_agent, chi_square_analyst_agent),
    ]
)

In [26]:
# main.py
async def run_analysis():
    # Initialize framework session services
    session_service = InMemorySessionService()
    
    await session_service.create_session(
        app_name="DataAnalystApp",
        user_id="user_123",
        session_id="session_001"
    )
    
    # Initialize runtime runner
    runner = Runner(
        agent=root_agent,
        app_name="DataAnalystApp",
        session_service=session_service
    )
    
    # Define task and target file
    user_query = (
        "We have a file named 'sales_data_sample.csv'. "
        "What columns are in the csv data and what are the top trends? "
        "Please find the statistical signifance of association between variables: 'STATUS' and 'DEALSIZE', using Chi-Square test. "
    )

    content = types.Content(role="user", parts=[types.Part(text=user_query)])
    
    print(f"User Request: {user_query}\n")
    print("--- Agent Processing ---")
    
    # Run the agent execution thread
    events = runner.run(
        user_id="user_123",
        session_id="session_001",
        new_message=content
    )
    
    # Iterate through streaming events to get the evaluation
    for event in events:
        if event.is_final_response():
            final_text = event.content.parts[0].text
            print("\nFinal Agent Answer:")
            print(final_text)

if __name__ == "__main__":
    #asyncio.run(run_analysis())
    await run_analysis()


User Request: We have a file named 'sales_data_sample.csv'. What columns are in the csv data and what are the top trends? Please find the statistical signifance of association between variables: 'STATUS' and 'DEALSIZE', using Chi-Square test. 

--- Agent Processing ---

Final Agent Answer:
/*PLANNING*/
1. Use the `load_csv` tool to load the 'sales_data_sample.csv' file and examine its columns, data types, and contents.
2. Identify the columns present in the CSV file.
3. Analyze the data to find top trends (such as top products, sales by status, sales by deal size, etc.).
4. Calculate the Chi-Square test of independence between 'STATUS' and 'DEALSIZE' to find the statistical significance of their association. Since we only have `load_csv`, we will examine the data, and if the dataset is large, we can process it using Python code internally or analyze a representative sample, or perform the calculations. Let's first inspect the data.

Let's invoke `load_csv` on 'sales_data_sample.csv'./*